#### Simple Gen AI APP Using Langchain

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ['OPENAI_API_KEY']=os.getenv("OPENAI_API_KEY")
## Langsmith Tracking
os.environ["LANGCHAIN_API_KEY"]=os.getenv("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_TRACING_V2"]="true"
os.environ["LANGCHAIN_PROJECT"]=os.getenv("LANGCHAIN_PROJECT")

In [2]:
## Data Ingestion--From the website we need to scrape the data
from langchain_community.document_loaders import WebBaseLoader

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [3]:
loader=WebBaseLoader("https://docs.smith.langchain.com/evaluation/tutorials/rag")
loader

In [4]:
docs=loader.load()
docs

[Document(metadata={'source': 'https://docs.smith.langchain.com/evaluation/tutorials/rag', 'title': 'Evaluate a RAG application - Docs by LangChain', 'language': 'en'}, page_content='Evaluate a RAG application - Docs by LangChainOur new LangChain Academy course on Deep Agents is now live! Enroll for free.Docs by LangChain home pagePythonSearch...⌘KLangSmithPlatform for LLM observability and evaluationOverviewConceptsEvaluation approachesDatasetsCreate a datasetManage datasetsSet up evaluationsRun an evaluationEvaluation typesFrameworks & integrationsEvaluation techniquesImprove evaluatorsTutorialsEvaluate a chatbotEvaluate a RAG applicationTest a ReAct agent with Pytest/Vitest and LangSmithEvaluate a complex agentRun backtests on a new version of an agentAnalyze experiment resultsAnalyze an experimentCompare experiment resultsFilter experiments in the UIFetch performance metrics for an experimentUpload experiments run outside of LangSmithAnnotation & human feedbackUse annotation queues

In [5]:
### Load Data--> Docs-->Divide our Docuemnts into chunks dcouments-->text-->vectors-->Vector Embeddings--->Vector Store DB
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
documents=text_splitter.split_documents(docs)

In [6]:
documents

[Document(metadata={'source': 'https://docs.smith.langchain.com/evaluation/tutorials/rag', 'title': 'Evaluate a RAG application - Docs by LangChain', 'language': 'en'}, page_content='Evaluate a RAG application - Docs by LangChainOur new LangChain Academy course on Deep Agents is now live! Enroll for free.Docs by LangChain home pagePythonSearch...⌘KLangSmithPlatform for LLM observability and evaluationOverviewConceptsEvaluation approachesDatasetsCreate a datasetManage datasetsSet up evaluationsRun an evaluationEvaluation typesFrameworks & integrationsEvaluation techniquesImprove evaluatorsTutorialsEvaluate a chatbotEvaluate a RAG applicationTest a ReAct agent with Pytest/Vitest and LangSmithEvaluate a complex agentRun backtests on a new version of an agentAnalyze experiment resultsAnalyze an experimentCompare experiment resultsFilter experiments in the UIFetch performance metrics for an experimentUpload experiments run outside of LangSmithAnnotation & human feedbackUse annotation queues

In [7]:
from langchain_openai import OpenAIEmbeddings
embeddings=OpenAIEmbeddings()

In [8]:
from langchain_community.vectorstores import FAISS
vectorstoredb=FAISS.from_documents(documents,embeddings)

In [9]:
vectorstoredb

In [11]:
## Query From a vector db
query="What is RAG"
result=vectorstoredb.similarity_search(query)
result[0].page_content

'Retrieval Augmented Generation (RAG) is a technique that enhances Large Language Models (LLMs) by providing them with relevant external knowledge. It has become one of the most widely used approaches for building LLM applications.\nThis tutorial will show you how to evaluate your RAG applications using LangSmith. You’ll learn:'

In [12]:
from langchain_openai import ChatOpenAI
llm=ChatOpenAI(model="gpt-4o")

In [13]:
## Retrieval Chain, Document chain

from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

prompt=ChatPromptTemplate.from_template(
    """
Answer the following question based only on the provided context:
<context>
{context}
</context>


"""
)

document_chain=create_stuff_documents_chain(llm,prompt)
document_chain

RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='\nAnswer the following question based only on the provided context:\n<context>\n{context}\n</context>\n\n\n'), additional_kwargs={})])
| ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x0000018658F97310>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x0000018658F96CB0>, root_client=<openai.OpenAI object at 0x000001863DDAB820>, root_async_client=<openai.AsyncOpenAI object at 0x0000018658F95720>, model_name='gpt-4o', model_kwargs={}, openai_api_key=SecretStr('**********'))
| StrOutputParser(), kwargs={}, confi

In [16]:
from langchain_core.documents import Document
document_chain.invoke({
    "input":"What is RAG",
    "context":[Document(page_content=""" How to create test datasets
How to run your RAG application on those datasets
How to measure your application’s performance using different evaluation metrics""")]
})

'The provided context outlines three main steps in working with test datasets for a RAG (Retrieval-Augmented Generation) application:\n\n1. **Creating Test Datasets**: This involves generating or collecting datasets that will be used to evaluate the performance of your RAG application. These datasets should be representative of the tasks or queries your application is designed to handle.\n\n2. **Running Your RAG Application on Those Datasets**: This step involves deploying and executing your RAG application using the test datasets. This allows you to observe how the application processes the inputs and generates outputs, which is crucial for the subsequent evaluation.\n\n3. **Measuring Your Application’s Performance**: After running your application on the test datasets, you need to evaluate its performance using various evaluation metrics. Common metrics might include accuracy, precision, recall, F1 score, or other criteria that are relevant to the specific application and tasks at ha

However, we want the documents to first come from the retriever we just set up. That way, we can use the retriever to dynamically select the most relevant documents and pass those in for a given question.

In [17]:
### Input--->Retriever--->vectorstoredb

vectorstoredb

In [18]:
retriever=vectorstoredb.as_retriever()
from langchain.chains import create_retrieval_chain
retrieval_chain=create_retrieval_chain(retriever,document_chain)


In [19]:
retrieval_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x0000018658F944F0>, search_kwargs={}), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | ChatPromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='\nAnswer the following question based only on the provided context:\n<context>\n{context}\n</context>\n\n\n'), additional_kwargs={})])
            | 

In [20]:
## Get the response form the LLM
response=retrieval_chain.invoke({"input":"LangSmith has two usage limits: total traces and extended"})
response['answer']

'The context provided covers several topics related to the setup, evaluation, and functionality of Retrieval-Augmented Generation (RAG) applications using LangChain and LangSmith. It explains various components such as setting up datasets, evaluating applications, annotation, human feedback, and running evaluations. It also mentions building a basic RAG application by indexing and retrieving content from a vector store and generating a response using a Large Language Model (LLM). Additionally, there are mentions of installing necessary dependencies and the usage of LangSmith for evaluation and observability in RAG applications. The provided section highlights the broad applicability of these techniques across different frameworks beyond just LangChain, encouraging the use of preferred tools and libraries. Furthermore, it hints at available tutorials and courses related to deep agents.'

In [21]:

response

{'input': 'LangSmith has two usage limits: total traces and extended',
 'context': [Document(id='c419fa86-2c89-4cba-b87d-e2972aa1d139', metadata={'source': 'https://docs.smith.langchain.com/evaluation/tutorials/rag', 'title': 'Evaluate a RAG application - Docs by LangChain', 'language': 'en'}, page_content='annotation queuesSet up feedback criteriaAnnotate traces and runs inlineAudit evaluator scoresCommon data typesExample data formatDataset prebuilt JSON schema typesDataset transformationsOur new LangChain Academy course on Deep Agents is now live! Enroll for free.Docs by LangChain home pagePythonSearch...⌘KGitHubForumForumSearch...NavigationTutorialsEvaluate a RAG applicationGet startedObservabilityEvaluationPrompt engineeringSelf-hostingAdministrationGet startedObservabilityEvaluationPrompt engineeringSelf-hostingAdministrationGitHubForumOn this pageOverviewSetupEnvironmentApplicationIndexing and retrievalGenerationDatasetEvaluatorsCorrectness: Response vs reference answerRelevance

In [ ]:
response['context']